# Minimum Silver Table 5: QASMBench Stabilizer Checks
**Course:** TU Delft DSAIT4000 (Data Management & Engineering) — Assignment 1  
**Target Table:** `silver/qasmbench/stabilizer_check.parquet`  
**Shared Provenance:** `results/part1/source_trace.parquet`


---

### Objectives & Contracts
1. **Schema Fidelity:** Build `silver/qasmbench/stabilizer_check.parquet` where **one row represents one parity/stabilizer check identified in a circuit**.
2. **Strict PyArrow Types:**
    - `source_record_id`: `string` (stable link to the statements that define the check)
    - `circuit_id`: `string` (link to the Silver circuit)
    - `check_id`: `string` (stable identifier within the circuit)
    - `ancilla_qubit`: `string` (ancilla/check qubit)
    - `data_qubits`: `list<string>` (data qubits participating in the parity check)
    - `syndrome_bit`: `string` (classical bit receiving the check result)
3. **Shared Tracing Rule:** Append QASMBench stabilizer-check records to `results/part1/source_trace.parquet` using `save_source_traces`.
4. **Dual-Lake Persistence:** Write Parquet locally and synchronize it to MinIO bucket `quantum-lake`.

**Background Definitions**

* **Check:** A logical parity condition indicating whether a set of qubits has even or odd parity, usually used for error correction.
* **Ancilla Qubit:** An extra helper qubit used during a quantum circuit, aiding in parity checks.
* **Syndrome Bit:** Stores the result of a parity check, indicating whether the check passed or failed, whether the set of data qubits had even or odd parity, and what error syndrome was detected.


In [1]:
import hashlib
import io
from pathlib import Path
import zipfile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Platform helpers from starter package
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.tracing import save_source_traces

# Detect if running in container (/workspace) or locally
BASE_DIR = Path("/workspace") if Path("/workspace").exists() else Path(".").resolve()
print(f"Base Directory: {BASE_DIR}")

# Load configuration
settings = Settings.from_environment()
print(f"Lake Backend: {settings.lake_backend}")
print(f"MinIO Endpoint: {settings.s3_endpoint} (Bucket: {settings.s3_bucket})")

Base Directory: /workspace
Lake Backend: minio
MinIO Endpoint: http://minio:9000 (Bucket: quantum-lake)


### Step 1: Bronze ingestion and stabilizer-check extraction

The extractor handles both source circuits with custom gate definitions and transpiled circuits with explicit CNOT statements. A check is represented by an ancilla that receives CNOTs from at least two distinct data qubits and is later measured into a classical bit.

In [2]:
# Locate the QASMBench Bronze zip object (try MinIO first, fallback to local path)
bronze_object_name = "bronze/source=qasmbench/qasmbench-qec.zip"
bronze_bytes = None

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        print(f"Fetching '{bronze_object_name}' from MinIO...")
        response = client.get_object(settings.s3_bucket, bronze_object_name)
        bronze_bytes = response.read()
        response.close()
        response.release_conn()
        print(f"Successfully retrieved from MinIO ({len(bronze_bytes):,} bytes)")
    except Exception as e:
        print(f"MinIO fetch warning: {e}. Falling back to local file.")

if bronze_bytes is None:
    candidates = [
        Path("/course-data/raw/source=qasmbench/qasmbench-qec.zip"),
        BASE_DIR.parent / "datasets/student-bundle/core/raw/source=qasmbench/qasmbench-qec.zip",
        Path("datasets/student-bundle/core/raw/source=qasmbench/qasmbench-qec.zip"),
    ]
    for path in candidates:
        if path.exists():
            print(f"Reading from local path: {path}")
            bronze_bytes = path.read_bytes()
            break

assert bronze_bytes is not None, "Could not locate the QASMBench archive!"
bronze_sha256 = hashlib.sha256(bronze_bytes).hexdigest()
print(f"QASMBench Bronze Archive SHA-256: {bronze_sha256}")
print(f"QASMBench Bronze Archive Size:    {len(bronze_bytes):,} bytes")

Fetching 'bronze/source=qasmbench/qasmbench-qec.zip' from MinIO...
Successfully retrieved from MinIO (144,172 bytes)
QASMBench Bronze Archive SHA-256: 60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e4f85722a
QASMBench Bronze Archive Size:    144,172 bytes


### Step 2: Feature extraction and stable lineage records

The parser keeps register-local qubit names, expands custom gate calls, and groups parity interactions by measured ancilla. Gate-definition bodies are not treated as independently executed statements.

In [3]:
import re

CNOT_STATEMENT = re.compile(r"^cx\s+([^,]+)\s*,\s*(.+)$", re.IGNORECASE)
REGISTER_DECLARATION = re.compile(r"^(qreg|creg)\s+(\w+)\s*\[\s*(\d+)\s*\]$")
GATE_HEADER = re.compile(r"gate\s+(\w+)\s*([^{}]*)\{(?P<body>.*?)\}", re.DOTALL)


def normalize_operand(operand):
    return "".join(operand.strip().split())


def parse_registers(text):
    registers = {}
    for statement in text.split(";"):
        statement = " ".join(statement.split())
        match = REGISTER_DECLARATION.fullmatch(statement)
        if match:
            kind, name, size = match.groups()
            registers[name] = (kind, int(size))
    return registers


def expand_operand(operand, registers):
    operand = normalize_operand(operand)
    match = re.fullmatch(r"(\w+)(?:\[(\d+)\])?", operand)
    assert match, f"Unsupported operand: {operand}"
    name, index = match.groups()
    assert name in registers, f"Unknown register: {name}"
    kind, size = registers[name]
    if index is not None:
        assert int(index) < size, f"Register index out of range: {operand}"
        return [f"{name}[{index}]"]
    return [f"{name}[{position}]" for position in range(size)]


def expand_pairs(left, right, registers):
    left_values = expand_operand(left, registers)
    right_values = expand_operand(right, registers)
    assert len(left_values) == len(right_values), f"CNOT registers do not align: {left}, {right}"
    return list(zip(left_values, right_values))


def parse_stabilizer_checks(member, text):
    cleaned = re.sub(r"//.*", "", text)
    registers = parse_registers(cleaned)
    measurement_map = {}
    for statement in cleaned.split(";"):
        statement = " ".join(statement.split())
        if not statement.startswith("measure "):
            continue
        measured, destination = [part.strip() for part in statement[len("measure "):].split("->", 1)]
        measured_values = expand_operand(measured, registers)
        destination_values = expand_operand(destination, registers)
        assert len(measured_values) == len(destination_values)
        measurement_map.update(zip(measured_values, destination_values))

    gate_definitions = {}
    for match in GATE_HEADER.finditer(cleaned):
        formal_names = [item.strip() for item in match.group(2).split(",") if item.strip()]
        body_pairs = []
        for statement in match.group("body").split(";"):
            statement = " ".join(statement.split())
            body_match = CNOT_STATEMENT.fullmatch(statement)
            if body_match:
                body_pairs.append((body_match.group(1).strip(), body_match.group(2).strip()))
        gate_definitions[match.group(1)] = (formal_names, body_pairs)

    executable_text = GATE_HEADER.sub("", cleaned)
    interactions = []
    for statement in executable_text.split(";"):
        statement = " ".join(statement.split())
        if not statement or statement.startswith(("OPENQASM", "include", "barrier", "opaque", "measure", "qreg", "creg")):
            continue
        tokens = statement.split(None, 1)
        if len(tokens) != 2:
            continue
        operation, operand_text = tokens
        if operation.lower() == "cx":
            left, right = [part.strip() for part in operand_text.split(",", 1)]
            interactions.extend(expand_pairs(left, right, registers))
        elif operation in gate_definitions:
            formal_names, body_pairs = gate_definitions[operation]
            actual_operands = [part.strip() for part in operand_text.split(",")]
            assert len(formal_names) == len(actual_operands), f"Custom gate arguments do not align: {statement}"
            substitutions = dict(zip(formal_names, actual_operands))
            for left, right in body_pairs:
                mapped_left = substitutions[left]
                mapped_right = substitutions[right]
                interactions.extend(expand_pairs(mapped_left, mapped_right, registers))

    by_ancilla = {}
    for data_qubit, ancilla_qubit in interactions:
        by_ancilla.setdefault(ancilla_qubit, set()).add(data_qubit)

    benchmark_name = Path(member).parent.name
    variant = "transpiled" if Path(member).stem.endswith("_transpiled") else "source"
    circuit_id = f"qasmbench:{member[:-5]}"
    records = []
    for check_number, ancilla_qubit in enumerate(sorted(by_ancilla)):
        data_qubits = sorted(by_ancilla[ancilla_qubit])
        syndrome_bit = measurement_map.get(ancilla_qubit)
        if len(data_qubits) < 2 or syndrome_bit is None:
            continue
        check_id = f"{circuit_id}:check:{check_number}"
        records.append({
            "source_record_id": f"qasmbench:{member}:check:{check_number}",
            "circuit_id": circuit_id,
            "check_id": check_id,
            "ancilla_qubit": ancilla_qubit,
            "data_qubits": data_qubits,
            "syndrome_bit": syndrome_bit,
        })
    return records


stabilizer_records = []
trace_records = []
with zipfile.ZipFile(io.BytesIO(bronze_bytes)) as archive:
    qasm_members = sorted(name for name in archive.namelist() if name.endswith(".qasm"))
    for member in qasm_members:
        records = parse_stabilizer_checks(member, archive.read(member).decode("utf-8"))
        stabilizer_records.extend(records)
        for record in records:
            trace_records.append({
                "source_record_id": record["source_record_id"],
                "source_name": "qasmbench",
                "bronze_object": bronze_object_name,
                "archive_member": member,
                "record_locator": record["check_id"],
                "input_sha256": bronze_sha256,
            })
        print(f"  {Path(member).parent.name}/{Path(member).stem}: {len(records)} stabilizer checks")

assert stabilizer_records, "No measured stabilizer checks were found"
assert len({record["source_record_id"] for record in stabilizer_records}) == len(stabilizer_records)
print(f"\nExtracted {len(stabilizer_records)} stabilizer checks")

  error_correctiond3_n5/error_correctiond3_n5: 1 stabilizer checks
  error_correctiond3_n5/error_correctiond3_n5_transpiled: 1 stabilizer checks
  qec_en_n5/qec_en_n5: 1 stabilizer checks
  qec_en_n5/qec_en_n5_transpiled: 1 stabilizer checks
  qec_sm_n5/qec_sm_n5: 2 stabilizer checks
  qec_sm_n5/qec_sm_n5_transpiled: 2 stabilizer checks

Extracted 8 stabilizer checks


### Step 3: Strict Arrow schema and Parquet export

The `data_qubits` field is stored as a real Arrow list of strings, not as a serialized Python list. This preserves the relational meaning of the parity participants.

In [4]:
stabilizer_check_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("circuit_id", pa.string()),
    ("check_id", pa.string()),
    ("ancilla_qubit", pa.string()),
    ("data_qubits", pa.list_(pa.string())),
    ("syndrome_bit", pa.string()),
])

table_checks = pa.Table.from_pandas(
    pd.DataFrame(stabilizer_records),
    schema=stabilizer_check_schema,
    preserve_index=False,
)
assert table_checks.schema.equals(stabilizer_check_schema)
print("=== QASMBench Stabilizer Check Table ===")
print(f"Rows: {table_checks.num_rows}, Columns: {table_checks.num_columns}")
print(table_checks.schema)

=== QASMBench Stabilizer Check Table ===
Rows: 8, Columns: 6
source_record_id: string
circuit_id: string
check_id: string
ancilla_qubit: string
data_qubits: list<item: string>
  child 0, item: string
syndrome_bit: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 821


### Step 4: Persist the Silver table and shared provenance

The final cell writes the Parquet table, synchronizes it to MinIO when configured, appends idempotent source traces, and reads the file back to verify its schema and row count.

In [5]:
silver_dir = BASE_DIR / "silver/qasmbench"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_parquet_path = silver_dir / "stabilizer_check.parquet"

results_dir = BASE_DIR / "results/part1"
results_dir.mkdir(parents=True, exist_ok=True)
trace_parquet_path = results_dir / "source_trace.parquet"

pq.write_table(table_checks, silver_parquet_path, compression="zstd")
print(f"Wrote Silver stabilizer-check table to: {silver_parquet_path} ({silver_parquet_path.stat().st_size:,} bytes)")

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        minio_key = "silver/qasmbench/stabilizer_check.parquet"
        client.fput_object(settings.s3_bucket, minio_key, str(silver_parquet_path))
        print(f"Uploaded Silver table to MinIO: {settings.s3_bucket}/{minio_key}")
    except Exception as e:
        print(f"Warning: MinIO upload failed: {e}")

trace_count = save_source_traces(
    new_records=trace_records,
    source_name="qasmbench",
    trace_file_path=trace_parquet_path,
    settings=settings,
)
print(f"Master source_trace table updated; total rows: {trace_count:,}")

saved = pq.read_table(silver_parquet_path)
assert saved.schema.equals(stabilizer_check_schema)
assert saved.num_rows == len(stabilizer_records)
assert saved.column("data_qubits").type == pa.list_(pa.string())
print("Table 5 validation passed.")

Wrote Silver stabilizer-check table to: /workspace/silver/qasmbench/stabilizer_check.parquet (5,095 bytes)
Uploaded Silver table to MinIO: quantum-lake/silver/qasmbench/stabilizer_check.parquet
Master source_trace table updated; total rows: 14
Table 5 validation passed.


In [8]:
import pyarrow.parquet as pq

circuit_path = "/workspace/silver/qasmbench/stabilizer_check.parquet"
trace_path = "/workspace/results/part1/source_trace.parquet"

circuits = pq.read_table(circuit_path)
traces = pq.read_table(trace_path)

print("Stabilizer checks table")
print("Rows:", circuits.num_rows)
print("Columns:", circuits.column_names)
print(circuits.schema)
print(circuits.to_pandas().to_string(index=False))

print("\nQASMBench traces")
trace_df = traces.to_pandas()
qasmbench_traces = trace_df[trace_df["source_name"] == "qasmbench"]
print("Rows:", len(qasmbench_traces))
print(qasmbench_traces.to_string(index=False))

Stabilizer checks table
Rows: 8
Columns: ['source_record_id', 'circuit_id', 'check_id', 'ancilla_qubit', 'data_qubits', 'syndrome_bit']
source_record_id: string
circuit_id: string
check_id: string
ancilla_qubit: string
data_qubits: list<element: string>
  child 0, element: string
syndrome_bit: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 821
                                                                   source_record_id                                                             circuit_id                                                                       check_id ancilla_qubit              data_qubits syndrome_bit
           qasmbench:small/error_correctiond3_n5/error_correctiond3_n5.qasm:check:1            qasmbench:small/error_correctiond3_n5/error_correctiond3_n5            qasmbench:small/error_correctiond3_n5/error_correctiond3_n5:check:1          q[2] [q[0], q[1], q[3], q[4]]         c[2]
qasmbench:small/error_c